# Final Analysis Pipeline — GELECTRA Emotion Inference (Colab)

Production inference notebook for Mechanism 3. Runs the fine-tuned German **GELECTRA** emotion classifier (Widmann & Wich, 2023) over the cleaned full corpus on a Colab GPU. For each article it loads the model from `models/final/`, cleans the text, truncates to the first 512 subword tokens, and scores the 8 discrete emotions (`anger`, `fear`, `disgust`, `sadness`, `joy`, `enthusiasm`, `pride`, `hope`). Output is the canonical emotion-scores CSV consumed by the analysis notebooks in [`../02_Emotion_Analysis/`](../02_Emotion_Analysis/). Mirrors `../03_emotion_pipeline.py` and shares helpers from `../emotion_utils.py`.

Use **Runtime → Change runtime type → GPU** for faster inference.

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

# emotion_utils.py, 03_emotion_pipeline.py, and models/ live next to this notebook.
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
# Install dependencies (same as requirements.txt)
req = PROJECT_ROOT / "requirements.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available(), "| Device count:", torch.cuda.device_count())

In [ ]:
# Download pytorch_model.bin (~445 MB) if missing
subprocess.check_call([sys.executable, str(PROJECT_ROOT / "02_download_model_weights.py")])

## Data path

- Set `DATA_CSV` in the next cell (e.g. `/content/df_combined.csv` after uploading the CSV to Colab).
- Or set `RUN_UPLOAD = True` in the cell below to upload `df_combined.csv` from your laptop.

In [ ]:
DATA_CSV = Path("/content/df_combined.csv")
OUTPUT_CSV = Path("/content/emotion_full_results.csv")

# Default: classify the Title column (512-token truncation at inference — same as CLI)
TEXT_COLUMN = "Text"
MIN_WORDS = 3
BATCH_SIZE = 32

In [ ]:
# Optional: set True to upload df_combined.csv from your laptop (updates DATA_CSV)
RUN_UPLOAD = False

if RUN_UPLOAD:
    try:
        from google.colab import files

        uploaded = files.upload()
        for name in uploaded:
            dest = Path("/content") / name
            dest.write_bytes(uploaded[name])
            DATA_CSV = dest
            print("Using", DATA_CSV)
    except ImportError:
        print("Not in Colab — set DATA_CSV manually.")
else:
    print("Skipping upload; using DATA_CSV =", DATA_CSV)

In [ ]:
_spec = importlib.util.spec_from_file_location(
    "emotion_pipeline_colab",
    PROJECT_ROOT / "03_emotion_pipeline.py",
)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

MODEL_DIR = PROJECT_ROOT / "models" / "final" / "german-nlp-group" / "electra-base-german-uncased"

result = _pipe.run_pipeline(
    data_path=DATA_CSV,
    model_dir=MODEL_DIR,
    output_path=OUTPUT_CSV,
    batch_size=BATCH_SIZE,
    text_column=TEXT_COLUMN,
    min_words=MIN_WORDS,
)
result.head()

In [ ]:
try:
    from google.colab import files

    files.download(str(OUTPUT_CSV))
except ImportError:
    print("Saved to:", OUTPUT_CSV)